# Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# Import libs
import os
from dotenv import load_dotenv
import chromadb
# embedding functions needed for ChromaDB
from chromadb.utils import embedding_functions

# for the evaluation tool
from typing import List, Any
from pydantic import BaseModel, Field
from lib.parsers import PydanticOutputParser

# for the web search tool
from tavily import TavilyClient

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [3]:
# Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")


assert os.getenv("OPENAI_API_KEY") is not None
assert os.getenv("TAVILY_API_KEY") is not None

OPENAI_BASE_URL = os.getenv(
    "OPENAI_BASE_URL",
    "https://openai.vocareum.com/v1"
)

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
# Create retrieve_game tool uses chroma client and collection created in part 1

# ChromaDB client
chroma_client = chromadb.PersistentClient(path="chromadb")
# embedding function
embedding_fn = embedding_functions.OpenAIEmbeddingFunction()
# collection of games
collection = chroma_client.get_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

@tool
def retrieve_game(query: str) -> list:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry. 

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...)
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    results = collection.query(query_texts=[query])
    return results["metadatas"][0] if results["metadatas"] else []

    

#### Evaluate Retrieval Tool

In [5]:
# Create evaluate_retrieval tool
class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the documents are useful to answer the question")
    description: str = Field(description="Description about the evaluation result")

@tool
def evaluate_retrieval(question: str, retrieved_docs: list) -> EvaluationReport:
    """
    Based on the user's question and on the list of retrieved documents, 
    it will analyze the usability of the documents to respond to that question. 
    args: 
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    # use the model to judge if the documents are enough to respond the user's question
    model = LLM(model="gpt-4o-mini")
    prompt = (
      "You are a helpful assistant that if the documents are enough to respond the user's question."
      f"Here is the user's question: {question}"
      f"Here is/are the retrieved documents: {retrieved_docs}"
      "Give a detailed explanation, so it's possible to take an action to accept it or not."
    )
    # ask the model to return the answer in the specified format
    response = model.invoke(prompt, response_format=EvaluationReport)
    # parse the response
    report = PydanticOutputParser(model_class=EvaluationReport).parse(response)
    return report


#### Game Web Search Tool

In [6]:
# Create game_web_search tool using the Tavily client to search the web
@tool
def game_web_search(question: str) -> dict:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - question: a question about game industry.
    """
    client = TavilyClient(api_key=TAVILY_API_KEY)  
    search_result = client.search(
        query=question,
        search_depth="advanced",
        include_answer=True,
        include_raw_content=False,
        include_images=False,
    )
    return {
        "answer": search_result.get("answer", ""),
        "results": search_result.get("results", []),
    }

### Agent

In [7]:
# Create Agent abstraction using StateMachine

# set of instructions 
instructions = """
You are UdaPlay, an AI research assistant for the video game industry.
Workflow for every question:
- Always call the retrieve_game tool to search the local game database.
- Do not answer question from memory alone.
- Always call evaluate_retrieval with the original question and the retrieved documents.
- If useful is true, answer from those documents.
- Only if useful is false, call game_web_search, then answer from the web results.
- Give a clear, structured answer and cite whether the source was the local DB or the web.
"""
udaplay_agent = Agent(
    # equip with an appropriate model
    model_name="gpt-4o-mini",
    instructions=instructions,
    # plug  Tools developed
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.0,  # to be more factual
)


In [11]:
# Invoke your agent over a set of three questions:
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

# the questions as a list
questions = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X realeased for Playstation 5?"
]


session_id = "session_1" # one session for multiple queries (rubric)

# need to reset the seassion otherwise the history is huge when re-running the cell
if session_id in udaplay_agent.memory.sessions: # if state already exists
    udaplay_agent.reset_session(session_id) # clear the state

previous_num_messages = 0 # track the messages from the previous question start with 0 

# loop over the questions and invoke the agent
for question in questions:
    run = udaplay_agent.invoke(question, session_id=session_id) 
    messages = run.get_final_state()["messages"]
    new_messages = messages[previous_num_messages:]
    previous_num_messages = len(messages)
    
    print("\n" + "-"*100 + "\n")
    print(f"Question: {question}")  
    
    # see the reasoning and tool usage in the prints
    print(f"\nReasoning and tool usage:")
    # loop over the new messages 
    for message in new_messages: 
        # skip the system message
        if message.role == "system":
            continue
        # getattr checks if the message has a tool_calls attribute
        if getattr(message, "tool_calls", None):
            for tool_call in message.tool_calls:
                print(f" -> Tool call {tool_call.function.name}({tool_call.function.arguments})")
        elif message.role == "tool":
            print(f" <- {message.name}: {str(message.content)[:200]}...")
        elif message.role == "assistant" and message.content:
            print(f"Assistant: {message.content}")
        else:
            continue # avoid printing the dotted line below
        print("." * 100)

    # see the final answer in the prints
    print(f"Final Answer: {messages[-1].content}")
    print("\n" + "-"*100 + "\n")


[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

----------------------------------------------------------------------------------------------------

Question: When Pokémon Gold and Silver was released?

Reasoning and tool usage:
 -> Tool call retrieve_game({"query":"Pokémon Gold and Silver release date"})
....................................................................................................
 <- retrieve_game: "[{'Platform': 'Game Boy Color', 'Genre': 'Role-playing', 'YearOfRelease': 1999, 'Publisher': 'Nintendo', 'Description': 'Second-generation Pok\u00e9mon games introducing new regions, Pok\u00e9mon, an...
...................................................................

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes